# WP3 — YOLOv8 Agent Chaussée (D20 / D40)

**Training: NOT YET EXECUTED** until the cells below run on a **CUDA GPU**.

This notebook does **not** contain Precision, Recall, mAP, F1, FPS, or model-size numbers. Metrics are written to `training/reports/wp3/` from Ultralytics after a real run. Do not paste placeholder accuracy into markdown.

| Rule | Value |
| --- | --- |
| Agent | WP3 pavement only |
| Classes | 0=D20 alligator, 1=D40 pothole |
| Depth | Forbidden |
| Split | Existing seed-42 hash split — **do not re-split** |
| Expected counts | train 7380 / val 1581 / test 1582 / 17160 boxes |
| CPU | **Forbidden** — stop if `torch.cuda.is_available()` is false |
| WP2 / WP4 | Out of scope |

Repo: `TrabelsiAmin/Road-damage-project` branch `feature/road-damage-ai-improvements`.


## 1. Setup


In [ ]:
# Colab: Runtime → Change runtime type → GPU (T4 or better).
import os, sys, subprocess
from pathlib import Path

%pip -q install -U ultralytics torch torchvision pyyaml pillow numpy matplotlib tqdm

print("pip done", flush=True)


## 2. GPU verification — stop without CUDA


In [ ]:
import torch

print("torch", torch.__version__)
print("cuda_built", torch.version.cuda)
print("cuda_available", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise SystemExit("CUDA GPU required for WP3 training.")

print("device", torch.cuda.get_device_name(0))
props = torch.cuda.get_device_properties(0)
print("total_memory_GB", round(props.total_memory / 1024**3, 2))
try:
    import ultralytics
    print("ultralytics", ultralytics.__version__)
except Exception as exc:
    raise SystemExit(f"ultralytics import failed: {exc}")


## 3. Clone repo and locate the pavement split

JPEGs are **gitignored**. After cloning, point `WP3_PAVEMENT_ROOT` at Drive or unzip `pavement_wp3_splits.zip`.

On the machine that already has the processed split (do **not** re-download the 13 GB RDD zip):

```bash
cd training/data/processed
zip -r pavement_wp3_splits.zip pavement
```


In [ ]:
from pathlib import Path
import os, sys, zipfile

REPO_URL = "https://github.com/TrabelsiAmin/Road-damage-project.git"
BRANCH = "feature/road-damage-ai-improvements"

cwd = Path.cwd()
if (cwd / "training" / "config" / "pavement.yaml").exists():
    REPO_ROOT = cwd
elif (cwd / "config" / "pavement.yaml").exists():
    REPO_ROOT = cwd.parent
else:
    dest = Path("/content/Road-damage-project")
    if not dest.exists():
        subprocess = __import__("subprocess")
        subprocess.check_call(["git", "clone", "-b", BRANCH, REPO_URL, str(dest)])
    REPO_ROOT = dest

TRAINING = REPO_ROOT / "training"
os.chdir(TRAINING)
sys.path.insert(0, str(TRAINING))
print("TRAINING", TRAINING.resolve())

# Optional Drive mount (no-op outside Colab)
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Drive mount skipped:", exc)

# Set this if the split lives on Drive, e.g. /content/drive/MyDrive/tariqmap/pavement
DRIVE_PAVEMENT = os.environ.get("WP3_PAVEMENT_ROOT", "")
ZIP_CANDIDATES = [
    Path("/content/drive/MyDrive/tariqmap/pavement_wp3_splits.zip"),
    Path("/content/pavement_wp3_splits.zip"),
]
UNZIP_DEST = Path("/content/wp3_data")

from src.verify_wp3_dataset import locate_pavement_root, CANDIDATE_ROOTS

root = Path(DRIVE_PAVEMENT) if DRIVE_PAVEMENT else locate_pavement_root()
if root is None or not (root / "images" / "train").is_dir():
    for z in ZIP_CANDIDATES:
        if z.exists():
            print("unzipping", z)
            UNZIP_DEST.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(UNZIP_DEST)
            # zip may contain pavement/ or images/
            for cand in [UNZIP_DEST / "pavement", UNZIP_DEST]:
                if (cand / "images" / "train").is_dir():
                    root = cand
                    break
            break

if root is None:
    raise SystemExit(
        "WP3 pavement split not found. Images are gitignored. "
        "Upload pavement_wp3_splits.zip or set WP3_PAVEMENT_ROOT to "
        "training/data/processed/pavement (7380/1581/1582 images)."
    )

WP3_ROOT = Path(root).resolve()
os.environ["WP3_PAVEMENT_ROOT"] = str(WP3_ROOT)
print("WP3_ROOT", WP3_ROOT)


## 4. Dataset verification (must pass; do not re-split)


In [ ]:
from src.verify_wp3_dataset import verify, write_colab_yaml, EXPECTED_COUNTS
import json

yaml_src = TRAINING / "config" / "pavement.yaml"
report = verify(WP3_ROOT, yaml_src, check_expected_counts=True)
(TRAINING / "reports" / "wp3").mkdir(parents=True, exist_ok=True)
(TRAINING / "reports" / "wp3" / "dataset_verify.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2)[:4000])
if report["status"] != "OK":
    raise SystemExit("WP3 dataset verification FAILED. Do not train. Do not re-split.")
print("expected", EXPECTED_COUNTS)
print("split_images", report["split_images"])
print("boxes", report["box_counts_by_name"])


## 5. Ground-truth size stats (measured if images readable; else NOT AVAILABLE)


In [ ]:
from src.eval_small_objects import analyze_root
import json

stats = analyze_root(WP3_ROOT, split=None)
(TRAINING / "reports" / "wp3" / "small_objects_gt.json").write_text(json.dumps(stats, indent=2))
print("note", stats.get("note"))
for split, block in stats.get("splits", {}).items():
    print(split, block.get("status"), block.get("per_class"))


## 6. YOLO YAML with absolute Colab path


In [ ]:
from src.verify_wp3_dataset import write_colab_yaml

COLAB_YAML = TRAINING / "reports" / "wp3" / "pavement.colab.yaml"
write_colab_yaml(WP3_ROOT, COLAB_YAML)
print(COLAB_YAML.read_text())


## 7. Baseline — YOLOv8n, imgsz 640, seed 42, 100 epochs

Output: `runs/wp3_baseline/`. Batch `-1` lets Ultralytics pick a size from VRAM. Early stopping uses the YAML patience.


In [ ]:
from src.train import train as wp3_train

wp3_train(
    agent="pavement",
    data_config=COLAB_YAML,
    weights="yolov8n.pt",
    epochs=100,
    imgsz=640,
    seed=42,
    aug_config=TRAINING / "config" / "augmentation.yaml",
    project=str(TRAINING / "runs"),
    allow_cpu=False,
    finetune=False,
    batch=-1,
    run_name="wp3_baseline",
)
print("baseline weights", TRAINING / "runs" / "wp3_baseline" / "weights" / "best.pt")


## 8. Evaluate the **fixed test** split — metrics from Ultralytics only


In [ ]:
from src.eval_wp3 import evaluate_wp3

BASE_WEIGHTS = TRAINING / "runs" / "wp3_baseline" / "weights" / "best.pt"
base_eval = evaluate_wp3(
    weights=BASE_WEIGHTS,
    data=COLAB_YAML,
    split="test",
    imgsz=640,
    output=TRAINING / "reports" / "wp3" / "baseline" / "metrics.json",
    run_name="wp3_baseline",
)
print("status", base_eval["status"])
print("metrics keys", None if not base_eval["metrics"] else list(base_eval["metrics"]))

FINAL_WEIGHTS = BASE_WEIGHTS  # replaced after fine-tune if that cell runs


## 9. Small-object bins on the test split (GT; model AP is only in the JSON above)


In [ ]:
from src.eval_small_objects import analyze_root
import json

so = analyze_root(WP3_ROOT, split="test")
(TRAINING / "reports" / "wp3" / "baseline" / "small_objects.json").write_text(json.dumps(so, indent=2))
print(json.dumps(so.get("splits", {}).get("test", {}), indent=2)[:3000])


## 10. Controlled experiments (one change at a time)

| ID | Model | imgsz | Notes |
| --- | --- | ---: | --- |
| A | YOLOv8n | 640 | Same as baseline (skip if baseline already logged) |
| B | YOLOv8s | 640 | Larger backbone, same image size |
| C | YOLOv8s | 960 | Only if VRAM allows; skip otherwise |


In [ ]:
from pathlib import Path
import torch
import json
from src.train import train as wp3_train
from src.eval_wp3 import evaluate_wp3

VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
experiments = [
    {"id": "A", "name": "wp3_expA_n640", "weights": "yolov8n.pt", "imgsz": 640, "run": False},
    {"id": "B", "name": "wp3_expB_s640", "weights": "yolov8s.pt", "imgsz": 640, "run": True},
    {"id": "C", "name": "wp3_expC_s960", "weights": "yolov8s.pt", "imgsz": 960, "run": VRAM_GB >= 14},
]
print("VRAM_GB", round(VRAM_GB, 2), "will_run_C", VRAM_GB >= 14)

exp_summaries = []
for exp in experiments:
    rec = dict(exp)
    rec["status"] = "SKIPPED"
    rec["metrics"] = None
    if not exp["run"]:
        rec["reason"] = "skipped (A duplicates baseline or C needs more VRAM)"
        exp_summaries.append(rec)
        continue
    wp3_train(
        agent="pavement",
        data_config=COLAB_YAML,
        weights=exp["weights"],
        epochs=100,
        imgsz=exp["imgsz"],
        seed=42,
        aug_config=TRAINING / "config" / "augmentation.yaml",
        project=str(TRAINING / "runs"),
        allow_cpu=False,
        finetune=False,
        batch=-1,
        run_name=exp["name"],
    )
    w = TRAINING / "runs" / exp["name"] / "weights" / "best.pt"
    ev = evaluate_wp3(
        weights=w,
        data=COLAB_YAML,
        split="test",
        imgsz=exp["imgsz"],
        output=TRAINING / "reports" / "wp3" / exp["name"] / "metrics.json",
        run_name=exp["name"],
    )
    rec["status"] = ev["status"]
    rec["metrics"] = ev.get("metrics")
    rec["weights"] = str(w)
    exp_summaries.append(rec)

(TRAINING / "reports" / "wp3" / "experiments.json").write_text(json.dumps(exp_summaries, indent=2))
print([(e["id"], e["status"]) for e in exp_summaries])


## 11. Optional fine-tune of the justified candidate

Uses the **same test split**. Pick the run with the best D20/D40 AP50 **and** acceptable size — not automatically YOLOv8s-960.


In [ ]:
from src.train import train as wp3_train
from src.eval_wp3 import evaluate_wp3
import json

def _map50(ev):
    m = (ev or {}).get("metrics") or {}
    return m.get("mAP50")

candidates = [("wp3_baseline", base_eval, 640, BASE_WEIGHTS)]
for e in exp_summaries:
    if e.get("status") == "OK" and e.get("weights"):
        candidates.append((e["name"], {"metrics": e.get("metrics"), "status": e["status"]}, e["imgsz"], Path(e["weights"])))

ranked = sorted(
    [c for c in candidates if c[1].get("status") == "OK" and _map50(c[1]) is not None],
    key=lambda c: _map50(c[1]),
    reverse=True,
)
print("ranked", [(c[0], _map50(c[1])) for c in ranked])
if not ranked:
    raise SystemExit("No OK eval JSON yet — cannot fine-tune. This is not a fake result.")

best_name, best_ev, best_imgsz, best_w = ranked[0]
print("fine-tune init", best_name, best_w)

wp3_train(
    agent="pavement",
    data_config=COLAB_YAML,
    weights=str(best_w),
    epochs=50,
    imgsz=best_imgsz,
    seed=42,
    aug_config=TRAINING / "config" / "augmentation.yaml",
    project=str(TRAINING / "runs"),
    allow_cpu=False,
    finetune=True,
    batch=-1,
    run_name="wp3_improved",
)

FINAL_WEIGHTS = TRAINING / "runs" / "wp3_improved" / "weights" / "best.pt"
final_eval = evaluate_wp3(
    weights=FINAL_WEIGHTS,
    data=COLAB_YAML,
    split="test",
    imgsz=best_imgsz,
    output=TRAINING / "reports" / "wp3" / "final" / "metrics.json",
    run_name="wp3_improved",
)
print("final status", final_eval["status"])


## 12. Compare baseline vs improved — no “better” unless both JSON files are OK


In [ ]:
from evaluation.compare_wp3_models import compare

cmp = compare(
    TRAINING / "reports" / "wp3" / "baseline" / "metrics.json",
    TRAINING / "reports" / "wp3" / "final" / "metrics.json",
    TRAINING / "reports" / "wp3" / "compare.json",
)
print(cmp["markdown_table"])
print("declare_better", cmp["declare_better"])
print(cmp["selection_note"])


## 13. Selection criteria (read, then look at the table)

1. D40 AP50 and recall (potholes, many small boxes in the dump)  
2. D20 AP50  
3. Small-object GT bins vs any available per-class AP  
4. Checkpoint size and Ultralytics `speed_ms`  
5. **Not** “use the largest model”


## 14. Knowledge distillation stub — does not invent KD metrics


In [ ]:
from src.distill import plan

kd = plan(
    agent="pavement",
    teacher=TRAINING / "runs" / "wp3_expB_s640" / "weights" / "best.pt",
    student=Path("yolov8n.pt"),
    data=COLAB_YAML,
    output=TRAINING / "reports" / "wp3" / "kd_plan.json",
)
print("KD status", kd["status"], "metrics", kd["metrics"])
print(kd["reason"])


## 15. Export TFLite + verify (no dummy graph)


In [ ]:
from src.export import export_tflite
from src.verify_tflite import inspect_tflite
import json

export_dir = TRAINING / "reports" / "wp3" / "export"
try:
    tflite_path = export_tflite(
        weights_path=FINAL_WEIGHTS if FINAL_WEIGHTS.exists() else BASE_WEIGHTS,
        agent="pavement",
        output_dir=export_dir,
        precision="float16",
        imgsz=640,
    )
    print("exported", tflite_path, tflite_path.stat().st_size)
except SystemExit as exc:
    print("export NOT_RUN:", exc)
    tflite_path = export_dir / "pavement.tflite"

verify = inspect_tflite(tflite_path, expected_classes=2)
(TRAINING / "reports" / "wp3" / "tflite_verify.json").write_text(json.dumps(verify, indent=2))
print(json.dumps(verify, indent=2))

# Optional PyTorch vs TFLite smoke on one test image — skip if no interpreter
sample = next((WP3_ROOT / "images" / "test").glob("*.jpg"), None)
print("sample_image", sample)
if sample and tflite_path.exists() and verify.get("status") not in {"MODEL_PENDING"}:
    from ultralytics import YOLO
    pt = YOLO(str(FINAL_WEIGHTS if FINAL_WEIGHTS.exists() else BASE_WEIGHTS))
    pt_out = pt.predict(str(sample), imgsz=640, verbose=False)
    print("pytorch_n_boxes", 0 if not pt_out else len(pt_out[0].boxes))
    print("tflite_compare_note", "Run a real interpreter comparison only after verify status OK. Do not invent agreement metrics.")
else:
    print("pytorch_vs_tflite", "NOT_RUN")


## 16. Zip artefacts (not the dataset, not committed)


In [ ]:
from src.pack_wp3_artifacts import pack

manifest = pack(TRAINING, TRAINING / "wp3_training_artifacts.zip")
print("zip", TRAINING / "wp3_training_artifacts.zip")
print("status", manifest["status"], "files", manifest["file_count"], "weights", manifest["weight_files"])
print("Download the zip from Colab. Do not git-commit *.pt / *.tflite / this zip.")


## Done

If you interrupted before GPU training, the repo is still honest: **Training NOT YET EXECUTED**. Copy `reports/wp3/**/*.json` back to GitHub only when `status` is `OK`. Never hand-edit mAP into markdown.
